# 03 Mix Judger Policy

Notebook 02 searched module-wise G/A/S coefficients directly.  This notebook
turns those search traces into a reusable judger: a small model that scores
candidate mixtures from current round features and selects the best body/head/MoE
mix automatically.


In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path('/app/Object_Detection')
PROJECT = ROOT / 'dynamic_quality_aware_classwise_aggregation' / 'moe_dqa_judger'
OUT = PROJECT / 'output' / '03_mix_judger_policy'
OUT


## Train, Select, And Evaluate

The full evaluation is intentionally kept here because the goal is not only to
fit a predictor, but to verify whether its chosen coefficients behave like the
offline optimizer.


In [ ]:
import subprocess, sys

cmd = [
    sys.executable,
    str(PROJECT / 'scripts' / 'run_03_train_mix_judger.py'),
    '--workspace-root', str(OUT),
    '--rounds', '1,2,3,4,5',
    '--pool-samples', '2200',
    '--observed-templates', '12',
    '--val-batch-size', '32',
    '--evaluate-full',
    '--force',
]
print(' '.join(cmd))
subprocess.run(cmd, cwd=ROOT, check=True)


## Summaries


In [ ]:
selected = pd.read_csv(OUT / 'stats' / '03_selected_weights.csv')
display(selected[['round','candidate_id','guard_reason','pred_score','body_g','body_a','body_s','head_g','head_a','head_s','moe_g','moe_a','moe_s','pool_size']])

eval_path = OUT / 'stats' / '03_selected_full_eval.csv'
if eval_path.exists():
    full_eval = pd.read_csv(eval_path)
    display(full_eval[['round','map50','map50_95','precision','recall','score','body_g','body_a','body_s','head_g','head_a','head_s','moe_g','moe_a','moe_s']])

cv = pd.read_csv(OUT / 'stats' / '03_leave_one_round_cv.csv')
display(cv)


In [ ]:
print((OUT / '03_mix_judger_policy_report.md').read_text())
